# Text Stats And Formatter Agent

In this notebook, I build a small agent that can measure a piece of text and reshape it, using the Hugging Face `smolagents` library.

Like the calculator and converter agent in the previous notebook, this one does not look anything up. It answers questions about text by calling tools that count and reformat exactly, instead of asking the language model to count words in its head, which it tends to get wrong on longer text.

In this notebook, I will learn how to:

- Write a plain Python function that reports word, character and sentence counts
- Write a plain Python function that reshapes text into a different case
- Turn each one into an agent tool with the `@tool` decorator
- Reject an unknown mode inside a tool instead of guessing what it means
- Give an agent both tools and watch it choose, or chain, the right one

Everything here stays small on purpose, so the whole idea fits in one sitting.

## 1. Importing Libraries and Creating the Model

First I import the pieces I need from `smolagents`.

`CodeAgent` is the agent that writes and runs Python code to solve a task. `tool` is the decorator I use to turn a plain function into something an agent can call. `InferenceClientModel` is the language model, which runs on the Hugging Face Inference API rather than on my own machine.

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

## 2. Writing a Basic Text Stats Function

Before I build a tool, I write the counting as an ordinary Python function.

It takes a piece of text and a mode as a plain string, like `"words"` or `"characters"`, and returns a single number. Keeping the logic in a plain function first means I can test the counting on its own, without an agent or a language model anywhere near it.

In [ ]:
def text_stats(text: str, mode: str) -> int:
    """Returns one statistic about a piece of text."""
    if mode == "words":
        return len(text.split())
    if mode == "characters":
        return len(text)
    if mode == "sentences":
        return len([s for s in text.split(".") if s.strip()])
    raise ValueError(f"Unknown mode: {mode}")

## 3. Testing the Text Stats Function

I try the function on a sentence I can check by eye before trusting it with anything else. This is the easiest place to catch a mistake, because there is no agent involved yet, just plain Python I can read line by line.

In [ ]:
sentence = "The quick brown fox jumps over the lazy dog. It ran fast."

print(text_stats(sentence, "words"))
print(text_stats(sentence, "characters"))
print(text_stats(sentence, "sentences"))

## 4. Handling an Unknown Mode

One thing can go wrong with this function: passing a mode I did not plan for, like `"paragraphs"`. Right now, that already raises a `ValueError`, but the message does not say which modes are actually allowed.

I want the error to spell out the valid options, because this message is exactly what the tool will hand back to the agent later. A confusing error here becomes a confusing tool result there.

In [ ]:
def text_stats(text: str, mode: str) -> int:
    """Returns one statistic about a piece of text."""
    if mode == "words":
        return len(text.split())
    if mode == "characters":
        return len(text)
    if mode == "sentences":
        return len([s for s in text.split(".") if s.strip()])
    raise ValueError(
        f"Unknown mode: {mode}. Use words, characters or sentences."
    )

## 5. Testing the Error Handling

I check the failure case directly, catching the error myself so the notebook keeps running and I can read the message that would reach the agent.

In [ ]:
try:
    text_stats(sentence, "paragraphs")
except ValueError as error:
    print(error)

## 6. Turning the Text Stats Function Into a Tool

The function works, but an agent cannot call a plain Python function. It needs a tool.

The `@tool` decorator does the conversion. `smolagents` reads the docstring to build the description the model sees, so I describe the mode argument carefully, including the exact words it should use.

In [ ]:
@tool
def text_stats_tool(text: str, mode: str) -> str:
    """
    Returns one statistic about a piece of text.

    Args:
        text (str): The text to measure.
        mode (str): One of "words", "characters" or "sentences".
    """
    try:
        result = text_stats(text, mode)
    except ValueError as error:
        return str(error)
    return str(result)

## 7. Testing the Tool on Its Own

Before handing the tool to an agent, I call it directly, the same way the agent would. If something is wrong here, I know the mistake is in my code and not in how the model is using it.

In [ ]:
print(text_stats_tool(sentence, "words"))
print(text_stats_tool(sentence, "sentences"))
print(text_stats_tool(sentence, "paragraphs"))

## 8. Creating an Agent With the Text Stats Tool

Now I give the tool to a `CodeAgent`. The agent reads the question, decides it needs to measure some text, writes a line of Python that calls `text_stats_tool`, and turns the result into a sentence.

In [ ]:
counter_agent = CodeAgent(
    tools=[text_stats_tool],
    model=model
)

## 9. Asking the Agent a Counting Question

I ask a question in plain English rather than handing over the text and mode directly, so the agent has to work out what to measure before it calls the tool.

In [ ]:
counter_agent.run(
    "How many words are in this sentence: "
    "'The quick brown fox jumps over the lazy dog'?"
)

## 10. Adding a Simple Text Formatter Function

A word count only gets me so far, so I write a second, unrelated function: a text formatter. It supports a small, fixed set of reshapings, upper case, lower case, title case and a reversed string, which is enough to show the idea without turning this into a full text processing library.

In [ ]:
def format_text(text: str, mode: str) -> str:
    """Reshapes a piece of text into a different case."""
    if mode == "upper":
        return text.upper()
    if mode == "lower":
        return text.lower()
    if mode == "title":
        return text.title()
    if mode == "reverse":
        return text[::-1]
    raise ValueError(
        f"Unknown mode: {mode}. Use upper, lower, title or reverse."
    )

## 11. Testing the Formatter Function

Same habit as before: I try it on text I can check by eye, before it goes anywhere near a tool or an agent.

In [ ]:
print(format_text("agents are useful", "upper"))
print(format_text("AGENTS ARE USEFUL", "lower"))
print(format_text("agents are useful", "title"))
print(format_text("agents are useful", "reverse"))

## 12. Testing the Formatter's Error Handling

I check the same kind of failure as before, an unrecognised mode, so I know the formatter fails as cleanly as the counter does.

In [ ]:
try:
    format_text("agents are useful", "snake_case")
except ValueError as error:
    print(error)

## 13. Turning the Formatter Into a Tool

Same pattern as the counter: wrap the function in `@tool`, and write a docstring that spells out exactly which mode names it understands, because that list is the only thing standing between the agent and a `ValueError`.

In [ ]:
@tool
def format_text_tool(text: str, mode: str) -> str:
    """
    Reshapes a piece of text into a different case.

    Args:
        text (str): The text to reshape.
        mode (str): One of "upper", "lower", "title" or "reverse".
    """
    try:
        result = format_text(text, mode)
    except ValueError as error:
        return str(error)
    return result

## 14. Testing the Formatter Tool

I run it directly one more time before trusting an agent with it.

In [ ]:
print(format_text_tool("agents are useful", "upper"))
print(format_text_tool("agents are useful", "reverse"))
print(format_text_tool("agents are useful", "snake_case"))

## 15. Combining Both Tools Into One Agent

Now I build one agent with both tools. I do not tell it which one to use for which question; it reads both tool descriptions and decides for itself, the same way the assistant agent in the previous notebook chose between the calculator and the converter.

In [ ]:
assistant_agent = CodeAgent(
    tools=[text_stats_tool, format_text_tool],
    model=model
)

## 16. Asking a Question That Needs the Counter

This question only involves measuring text, so I expect the agent to reach for `text_stats_tool` and leave the formatter alone.

In [ ]:
assistant_agent.run(
    "How many characters are in the sentence 'Agents call tools instead of guessing'?"
)

## 17. Asking a Question That Needs the Formatter

This time there is nothing to count, just text to reshape, so a well working agent should pick `format_text_tool` instead.

In [ ]:
assistant_agent.run(
    "Give me the sentence 'agents are useful' in upper case."
)

## 18. Asking a Question That Needs Both Tools

This question cannot be answered with one tool call. The agent has to count the words in a sentence and then reshape that same sentence, so I can see it chain two tool calls together to reach one answer.

In [ ]:
assistant_agent.run(
    "Count the words in the sentence 'agents are useful', then give me "
    "that same sentence in title case."
)

## 19. Limitations of This Simple Agent

This agent is reliable for the exact modes I wrote, but it cannot do anything outside that fixed list.

If I ask it to convert text to snake_case, there is no mode for that. The call below fails cleanly with the error message I wrote, which is exactly the point: a missing mode should say so, not quietly return a made up answer. Adding snake_case later just means adding one more `if` branch to `format_text`, not making the tool description vaguer so the model can "figure it out".

In [ ]:
print(format_text_tool("agents are useful", "snake_case"))